# 🖥️ LLM Donanım Gereksinimleri Analizi

Bu notebook, GPT-2 ve benzeri Transformer modellerinin donanım gereksinimlerini hesaplar.

## İçerik
1. Model parametreleri hesaplama
2. Bellek gereksinimleri (VRAM, RAM)
3. GPU uyumluluk kontrolü
4. Eğitim süresi tahmini

---

## 1. Kütüphaneler ve Model Konfigürasyonları

In [ ]:
import pandas as pd
import torch

# Model konfigürasyonları (GPT-2 family)
MODEL_CONFIGS = {
    "GPT-2 Small (124M)": {
        "emb_dim": 768,
        "n_layers": 12,
        "n_heads": 12,
        "vocab_size": 50257,
        "context_length": 1024
    },
    "GPT-2 Medium (355M)": {
        "emb_dim": 1024,
        "n_layers": 24,
        "n_heads": 16,
        "vocab_size": 50257,
        "context_length": 1024
    },
    "GPT-2 Large (774M)": {
        "emb_dim": 1280,
        "n_layers": 36,
        "n_heads": 20,
        "vocab_size": 50257,
        "context_length": 1024
    },
    "GPT-2 XL (1558M)": {
        "emb_dim": 1600,
        "n_layers": 48,
        "n_heads": 25,
        "vocab_size": 50257,
        "context_length": 1024
    },
    # Ek modeller
    "LLaMA 7B": {
        "emb_dim": 4096,
        "n_layers": 32,
        "n_heads": 32,
        "vocab_size": 32000,
        "context_length": 2048
    },
    "LLaMA 13B": {
        "emb_dim": 5120,
        "n_layers": 40,
        "n_heads": 40,
        "vocab_size": 32000,
        "context_length": 2048
    }
}

# Popüler GPU'ların FP16 performansları
GPU_PERFORMANCE = {
    "RTX 4090 (24GB)": {"fp16_tflops": 82.58e12, "vram_gb": 24, "memory_bandwidth": 1.0e12},
    "RTX 3090 (24GB)": {"fp16_tflops": 35.58e12, "vram_gb": 24, "memory_bandwidth": 936e9},
    "RTX 3080 Ti (12GB)": {"fp16_tflops": 34.1e12, "vram_gb": 12, "memory_bandwidth": 912e9},
    "RTX 3080 (10GB)": {"fp16_tflops": 29.77e12, "vram_gb": 10, "memory_bandwidth": 760e9},
    "RTX 3070 (8GB)": {"fp16_tflops": 29.77e12, "vram_gb": 8, "memory_bandwidth": 448e9},
    "RTX 3060 Ti (8GB)": {"fp16_tflops": 16.2e12, "vram_gb": 8, "memory_bandwidth": 448e9},
    "RTX 3060 (12GB)": {"fp16_tflops": 12.74e12, "vram_gb": 12, "memory_bandwidth": 360e9},
    "GTX 1660 Super (6GB)": {"fp16_tflops": 5.0e12, "vram_gb": 6, "memory_bandwidth": 336e9},
    "A100 (80GB)": {"fp16_tflops": 77.97e12, "vram_gb": 80, "memory_bandwidth": 2.0e12},
    "A100 (40GB)": {"fp16_tflops": 77.97e12, "vram_gb": 40, "memory_bandwidth": 1.6e12},
    "H100 (80GB)": {"fp16_tflops": 204.9e12, "vram_gb": 80, "memory_bandwidth": 3.35e12},
    "A10G (24GB)": {"fp16_tflops": 31.24e12, "vram_gb": 24, "memory_bandwidth": 600e9},
    "T4 (16GB)": {"fp16_tflops": 65.13e12, "vram_gb": 16, "memory_bandwidth": 320e9},
}

print("✅ Model konfigürasyonları yüklendi")
print(f"   Toplam {len(MODEL_CONFIGS)} model")
print(f"   Toplam {len(GPU_PERFORMANCE)} GPU")

---

## 2. Model Parametre Hesaplama

In [ ]:
def calculate_parameters(emb_dim, n_layers, n_heads, vocab_size=50257, context_length=1024):
    """
    Transformer model parametrelerini hesapla
    
    Formüller:
    - Token Embedding: vocab_size × emb_dim
    - Pos Embedding: context_length × emb_dim
    - Attention QKV: 3 × (emb_dim × emb_dim)
    - Attention OutProj: emb_dim × emb_dim
    - FFN Up: emb_dim × (emb_dim × 4)
    - FFN Down: (emb_dim × 4) × emb_dim
    - LayerNorm: 2 × emb_dim (scale + shift)
    """
    
    # Embedding katmanları
    tok_emb = vocab_size * emb_dim
    pos_emb = context_length * emb_dim
    
    # Attention
    qkv = 3 * (emb_dim * emb_dim)
    out_proj = emb_dim * emb_dim
    attention = qkv + out_proj
    
    # Feed Forward Network
    ffn_up = emb_dim * (emb_dim * 4)
    ffn_down = (emb_dim * 4) * emb_dim
    ffn = ffn_up + ffn_down
    
    # LayerNorm (pre-attention + pre-FFN)
    ln = 2 * emb_dim
    
    # Bir transformer block
    block = attention + ffn + ln
    
    # Tüm block'lar
    total_blocks = block * n_layers
    
    # Final LayerNorm
    final_ln = 2 * emb_dim
    
    # Output Head
    output_head = emb_dim * vocab_size
    
    # TOPLAM
    total = tok_emb + pos_emb + total_blocks + final_ln + output_head
    
    return {
        "token_embedding": tok_emb,
        "positional_embedding": pos_emb,
        "attention_per_block": attention,
        "ffn_per_block": ffn,
        "block_params": block,
        "total_blocks": total_blocks,
        "final_ln": final_ln,
        "output_head": output_head,
        "total_params": total
    }

# Tüm modeller için parametreleri hesapla
print("=" * 80)
print("📊 MODEL PARAMETRE SAYILARI")
print("=" * 80)

param_results = []
for name, cfg in MODEL_CONFIGS.items():
    params = calculate_parameters(
        cfg["emb_dim"], 
        cfg["n_layers"], 
        cfg["n_heads"],
        cfg["vocab_size"],
        cfg["context_length"]
    )
    
    param_results.append({
        "Model": name,
        "emb_dim": cfg["emb_dim"],
        "n_layers": cfg["n_layers"],
        "n_heads": cfg["n_heads"],
        "Parametre (M)": f"{params['total_params']/1e6:.1f}M",
        "Parametre (B)": f"{params['total_params']/1e9:.2f}B"
    })

df_params = pd.DataFrame(param_results)
print(df_params.to_string(index=False))
print("=" * 80)

---

## 3. Bellek (Memory) Gereksinimleri

In [ ]:
def calculate_memory_requirements(emb_dim, n_layers, n_heads, vocab_size=50257, context_length=1024):
    """
    Model bellek gereksinimlerini hesapla
    
    Precision başına bayt sayısı:
    - FP32: 4 bytes
    - FP16/BF16: 2 bytes  
    - INT8: 1 byte
    """
    params = calculate_parameters(emb_dim, n_layers, n_heads, vocab_size, context_length)
    total = params["total_params"]
    
    # Model ağırlıkları
    model_fp32 = (total * 4) / (1024**3)  # GB
    model_fp16 = (total * 2) / (1024**3)  # GB
    model_int8 = (total * 1) / (1024**3)   # GB
    
    # Training için ek bellek
    # - Gradients: model boyutu kadar
    # - Optimizer states (AdamW): model boyutu × 2
    train_fp32 = model_fp32 * 4  # model + grad + 2×optimizer
    train_fp16 = model_fp16 * 4
    
    # KV Cache (inference için)
    head_dim = emb_dim // n_heads
    kv_cache = (2 * context_length * n_layers * n_heads * head_dim * 2) / (1024**3)  # 2 = K + V
    
    # Inference toplam (model + KV cache)
    inference_fp16 = model_fp16 + kv_cache
    
    return {
        "model_fp32_gb": model_fp32,
        "model_fp16_gb": model_fp16,
        "model_int8_gb": model_int8,
        "train_fp32_gb": train_fp32,
        "train_fp16_gb": train_fp16,
        "kv_cache_gb": kv_cache,
        "inference_fp16_gb": inference_fp16
    }

# Bellek gereksinimlerini hesapla
print("\n" + "=" * 100)
print("💾 BELLEK GEREKSİNİMLERİ")
print("=" * 100)

memory_results = []
for name, cfg in MODEL_CONFIGS.items():
    mem = calculate_memory_requirements(
        cfg["emb_dim"],
        cfg["n_layers"],
        cfg["n_heads"],
        cfg["vocab_size"],
        cfg["context_length"]
    )
    
    memory_results.append({
        "Model": name,
        "FP32": f"{mem['model_fp32_gb']:.2f} GB",
        "FP16/BF16": f"{mem['model_fp16_gb']:.2f} GB",
        "INT8": f"{mem['model_int8_gb']:.2f} GB",
        "KV Cache": f"{mem['kv_cache_gb']:.2f} GB",
        "Inference (FP16)": f"{mem['inference_fp16_gb']:.2f} GB",
        "Training (FP16)": f"{mem['train_fp16_gb']:.2f} GB"
    })

df_mem = pd.DataFrame(memory_results)
print(df_mem.to_string(index=False))
print("=" * 100)

---

## 4. GPU Uyumluluk Kontrolü

In [ ]:
# Kullanıcının sistemini burada tanımlayabilir
USER_GPU = "RTX 3070 (8GB)"  # <-- KENDİ GPU'UNUZU YAZIN

def check_gpu_compatibility(model_name, cfg, gpu_name):
    """GPU uyumluluğunu kontrol et"""
    
    if gpu_name not in GPU_PERFORMANCE:
        return {"status": "unknown", "message": "GPU tanınmıyor"}
    
    gpu = GPU_PERFORMANCE[gpu_name]
    vram = gpu["vram_gb"]
    
    mem = calculate_memory_requirements(
        cfg["emb_dim"],
        cfg["n_layers"],
        cfg["n_heads"],
        cfg["vocab_size"],
        cfg["context_length"]
    )
    
    # Inference kontrolü
    inf_vram = mem["inference_fp16_gb"]
    if inf_vram <= vram:
        inf_status = "✅"
    else:
        inf_status = "❌"
    
    # Training kontrolü (batch size 1)
    train_vram = mem["train_fp16_gb"]
    # Minimum batch için ~2× overhead ekle
    train_needed = train_vram * 1.5
    if train_needed <= vram:
        train_status = "✅"
    elif train_needed <= vram * 2:
        train_status = "⚠️"  # Batch küçültmeli
    else:
        train_status = "❌"
    
    return {
        "vram": vram,
        "inf_needed": inf_vram,
        "inf_status": inf_status,
        "train_needed": train_needed,
        "train_status": train_status
    }

# Tüm GPU'lar için uyumluluk tablosu
print("\n" + "=" * 100)
print("✅ GPU UYUMLULUK MATRİSİ")
print("=" * 100)

compatibility_matrix = []

for gpu_name, gpu_info in GPU_PERFORMANCE.items():
    row = {"GPU": gpu_name, "VRAM": f"{gpu_info['vram_gb']}GB"}
    
    for model_name, cfg in MODEL_CONFIGS.items():
        comp = check_gpu_compatibility(model_name, cfg, gpu_name)
        short_model = model_name.split()[1] + " " + model_name.split()[2] if len(model_name.split()) > 2 else model_name.split()[1]
        
        # Inference durumu
        if comp["inf_status"] == "✅":
            row[short_model + " Inf"] = "✅"
        else:
            row[short_model + " Inf"] = "❌"
    
    compatibility_matrix.append(row)

# DataFrame oluştur
all_models = [m.split()[1] + " " + m.split()[2] if len(m.split()) > 2 else m.split()[1] for m in MODEL_CONFIGS.keys()]
cols = ["GPU", "VRAM"] + [m + " Inf" for m in all_models]

df_comp = pd.DataFrame(compatibility_matrix)
df_comp = df_comp[[c for c in cols if c in df_comp.columns]]
print(df_comp.to_string(index=False))

print("\n🔒 Not: Training için daha fazla VRAM gerekir (gradient + optimizer states)")
print("=" * 100)

---

## 5. Eğitim Süresi Tahmini

In [ ]:
def estimate_training_time(model_params, gpu_tflops, training_tokens=1e9, batch_size=8, seq_len=1024):
    """
    Tahmini eğitim süresini hesapla
    
    Args:
        model_params: Toplam parametre sayısı
        gpu_tflops: GPU FP16 TFLOPs
        training_tokens: Eğitim token sayısı (default 1B)
        batch_size: Batch boyutu
        seq_len: Sequence uzunluğu
    """
    
    # FLOPs per token (transformer için yaklaşık)
    # Forward: ~6P, Backward: ~6P, Total: ~12P
    flops_per_token = 12 * model_params
    
    # Tokens per batch
    tokens_per_batch = batch_size * seq_len
    
    # Toplam batch sayısı
    num_batches = training_tokens / tokens_per_batch
    
    # Toplam FLOPs
    total_flops = flops_per_token * training_tokens
    
    # MFU (Model FLOPs Utilization)
    # Training için tipik: 30-50%
    mfu = 0.40  # %40 ortalama
    effective_flops = gpu_tflops * mfu
    
    # Saniye
    seconds = total_flops / effective_flops
    
    return seconds / 3600  # Saate çevir

# Eğitim süreleri
print("\n" + "=" * 100)
print("⏱️ TAHMİNİ EĞİTİM SÜRESİ (1 Milyar Token, Batch Size 8)")
print("=" * 100)

training_results = []

for model_name, cfg in MODEL_CONFIGS.items():
    params = calculate_parameters(cfg["emb_dim"], cfg["n_layers"], cfg["n_heads"], cfg["vocab_size"], cfg["context_length"])
    
    row = {"Model": model_name}
    
    for gpu_name, gpu_info in GPU_PERFORMANCE.items():
        hours = estimate_training_time(
            params["total_params"],
            gpu_info["fp16_tflops"],
            training_tokens=1e9
        )
        
        if hours < 1:
            row[gpu_name.split()[0]] = f"{hours*60:.0f}dk"
        elif hours < 24:
            row[gpu_name.split()[0]] = f"{hours:.1f}sa"
        else:
            row[gpu_name.split()[0]] = f"{hours/24:.1f}g"
    
    training_results.append(row)

df_time = pd.DataFrame(training_results)
print(df_time.to_string(index=False))

print("\n📝 Notlar:")
print("   - Bu süreler 1 milyar token eğitim için tahminidir")
print("   - MFU=%40 varsayılarak hesaplanmıştır")
print("   - Gerçek süre model implementasyonuna bağlıdır")
print("=" * 100)

---

## 6. Özet Tablolar

In [ ]:
# Özet: En popüler GPU'lar için
print("\n" + "=" * 80)
print("📋 HIZLI BAŞVURU TABLOSU")
print("=" * 80)

popular_gpus = ["RTX 3060 (12GB)", "RTX 3070 (8GB)", "RTX 3080 (10GB)", "RTX 3090 (24GB)", "RTX 4090 (24GB)", "A100 (40GB)"]

summary = []
for model_name, cfg in list(MODEL_CONFIGS.items())[:4]:  # Sadece GPT-2 modelleri
    mem = calculate_memory_requirements(cfg["emb_dim"], cfg["n_layers"], cfg["n_heads"], cfg["vocab_size"], cfg["context_length"])
    
    # Hangi GPU'lar çalıştırabilir?
    compatible_gpus = []
    for gpu_name in popular_gpus:
        if gpu_name in GPU_PERFORMANCE:
            vram = GPU_PERFORMANCE[gpu_name]["vram_gb"]
            if mem["inference_fp16_gb"] <= vram:
                compatible_gpus.append(gpu_name.split()[0] + "-" + gpu_name.split()[1])
    
    summary.append({
        "Model": model_name,
        "Param": f"{calculate_parameters(cfg['emb_dim'], cfg['n_layers'], cfg['n_heads'], cfg['vocab_size'], cfg['context_length'])['total_params']/1e6:.0f}M",
        "VRAM (Inf)": f"{mem['inference_fp16_gb']:.1f}GB",
        "VRAM (Train)": f"{mem['train_fp16_gb']:.1f}GB",
        "Çalışan GPU'lar": ", ".join(compatible_gpus) if compatible_gpus else "Hiçbiri"
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))
print("=" * 80)

---

## 📝 Kullanım Önerileri

### Başlangıç (8GB VRAM ve altı)
- **124M model** - Mükemmel başlangıç
- Batch size: 4-8
- Mixed precision (FP16) kullanın

### Orta Seviye (8-12GB VRAM)
- **124M-355M model** - Araştırma ve fine-tuning
- Batch size: 4-8
- Gradient checkpointing düşünün

### İleri Seviye (24GB+ VRAM)
- **355M-774M model** - Full training
- Batch size: 16+
- Multi-GPU düşünün

### profesyonel (80GB+ VRAM)
- **7B+ model** - Large scale training
- Distributed training gerekli
- DeepSpeed / FSDP kullanın

## 🔧 Pratik İpuçları

| Teknik | Memory Tasarrufu | Açıklama |
|--------|------------------|-----------|
|**FP16/BF16**|2×|Model ağırlıkları 2 byte |
|**INT8**|4×|Quantization ile |
|**Gradient Accumulation**|Sınırsız|Effective batch artırır |
|**Gradient Checkpointing**|~30%|Compute memory trade-off |
|**KV Cache Offload**|VRAM→RAM|CPU RAM'e taşır |
|**Weight Tying**|~20%|Embedding = Output Head |

---

## Referanslar

- Sebastian Raschka - Build a Large Language Model From Scratch
- GPT-2: Language Models are Unsupervised Multitask Learners
- PaLM paper: Model FLOPs Utilization (MFU)

---
*Bu notebook kişisel öğrenme ve araştırma amaçlıdır.*